# Phase 4 — Meta-Learner Training & Evaluation

Trains two meta-learners (kNN, MLP) on each meta-feature representation and evaluates
with Leave-One-Out cross-validation (LOO-CV, n=51).

**Primary metric**: Top-1 accuracy — did the predicted best method actually rank first?  
**Secondary metric**: LSE-MAE — when predicting all 6 LSE values, mean absolute error  
**Baselines**:
- Lower bound: always predict k-means (most common, 14/51 = 27%)
- Upper bound: oracle (always correct, 100%)

**Ablation**: compares Option A (hand-crafted) vs B (autoencoder) vs C (dict learning)  
**Output**: saved models in `outputs/models/`

In [1]:
import os, sys, warnings, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import LeaveOneOut

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.join(ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, 'src'))

from meta_learner import (
    extract_Xy_clf, extract_Xy_reg,
    loo_classify, loo_regress,
    baseline_always, oracle_expected_lse,
    report_clf_results, LSE_COLS, METHOD_NAMES,
)

META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MODELS_DIR = os.path.join(ROOT, 'outputs', 'models')
FIGS_DIR   = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Imports OK')

Imports OK


In [2]:
# ── Load the three feature tables ─────────────────────────────────────────────
df_a = pd.read_csv(os.path.join(META_DIR, 'meta_training_optA.csv'))
df_b = pd.read_csv(os.path.join(META_DIR, 'meta_training_optB.csv'))
df_c = pd.read_csv(os.path.join(META_DIR, 'meta_training_optC.csv'))

# Drop rows where best_method is NaN (fully failed datasets)
df_a = df_a.dropna(subset=['best_method']).reset_index(drop=True)
df_b = df_b.dropna(subset=['best_method']).reset_index(drop=True)
df_c = df_c.dropna(subset=['best_method']).reset_index(drop=True)

print(f'Option A: {df_a.shape}  Option B: {df_b.shape}  Option C: {df_c.shape}')
print(f'Best method distribution (Option A):')
print(df_a['best_method'].value_counts().to_string())

Option A: (78, 27)  Option B: (78, 49)  Option C: (78, 49)
Best method distribution (Option A):
best_method
gmm          26
dbscan       21
kmeans       14
autoenc       9
agg           5
dictlearn     3


In [3]:
# ── Baselines ─────────────────────────────────────────────────────────────────
y_true = df_a['best_method'].values

baseline_kmeans = baseline_always('kmeans', y_true)
baseline_dbscan = baseline_always('dbscan', y_true)  # most frequent in our data
oracle_acc      = 1.0
oracle_lse      = oracle_expected_lse(df_a)

# Expected LSE if always picking k-means (regardless of whether it's best)
always_kmeans_lse = df_a['LSE_kmeans'].mean()

print('=== Baselines ===')
print(f'  Always k-means accuracy   : {baseline_kmeans:.3f}  (expected LSE: {always_kmeans_lse:.3f})')
print(f'  Always dbscan  accuracy   : {baseline_dbscan:.3f}')
print(f'  Oracle accuracy           : {oracle_acc:.3f}  (expected LSE: {oracle_lse:.3f})')

=== Baselines ===
  Always k-means accuracy   : 0.179  (expected LSE: 0.618)
  Always dbscan  accuracy   : 0.269
  Oracle accuracy           : 1.000  (expected LSE: 0.767)


## kNN — k Sweep on Option A

In [4]:
# ── Find best k via LOO-CV on Option A ────────────────────────────────────────
X_a, y_a, feat_cols_a, ids_a = extract_Xy_clf(df_a)

k_candidates = [1, 3, 5, 7, 9, 11]
k_results = {}

for k in k_candidates:
    pipe = Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('scale',  StandardScaler()),
        ('clf',    KNeighborsClassifier(n_neighbors=k)),
    ])
    res = loo_classify(pipe, X_a, y_a)
    k_results[k] = res['accuracy']
    print(f'  k={k:2d}  accuracy={res["accuracy"]:.3f}')

best_k = max(k_results, key=k_results.get)
print(f'\nBest k = {best_k}  (accuracy={k_results[best_k]:.3f})')

  k= 1  accuracy=0.590
  k= 3  accuracy=0.385
  k= 5  accuracy=0.423
  k= 7  accuracy=0.372
  k= 9  accuracy=0.359
  k=11  accuracy=0.385

Best k = 1  (accuracy=0.590)


## Classification LOO-CV — All Models × All Options

In [5]:
# ── Define final pipelines ────────────────────────────────────────────────────
knn_clf = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('clf',    KNeighborsClassifier(n_neighbors=best_k)),
])

mlp_clf = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('clf',    MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        max_iter=500,
        random_state=SEED,
    )),
])

print('Pipelines defined')

Pipelines defined


In [6]:
# ── Run classification LOO-CV for all 6 combos ────────────────────────────────
options = {
    'A_handcrafted': df_a,
    'B_autoencoder': df_b,
    'C_dictlearn'  : df_c,
}

clf_results = {}  # (option, model) -> result dict

for opt_name, df_opt in options.items():
    X_opt, y_opt, _, _ = extract_Xy_clf(df_opt)
    for model_name, pipe in [('kNN', knn_clf), ('MLP', mlp_clf)]:
        key = f'{opt_name} / {model_name}'
        print(f'Running {key} ...')
        res = loo_classify(pipe, X_opt, y_opt)
        clf_results[key] = res
        report_clf_results(key, res)

print('\nDone.')

Running A_handcrafted / kNN ...
  A_handcrafted / kNN                       Top-1 acc = 0.590  pred_dist={'dbscan': 20, 'agg': 5, 'dictlearn': 3, 'gmm': 29, 'autoenc': 9, 'kmeans': 12}
Running A_handcrafted / MLP ...
  A_handcrafted / MLP                       Top-1 acc = 0.564  pred_dist={'gmm': 29, 'agg': 4, 'dictlearn': 2, 'kmeans': 12, 'dbscan': 23, 'autoenc': 8}
Running B_autoencoder / kNN ...
  B_autoencoder / kNN                       Top-1 acc = 0.538  pred_dist={'dbscan': 22, 'kmeans': 16, 'gmm': 29, 'autoenc': 7, 'dictlearn': 2, 'agg': 2}
Running B_autoencoder / MLP ...
  B_autoencoder / MLP                       Top-1 acc = 0.513  pred_dist={'dbscan': 24, 'kmeans': 13, 'gmm': 31, 'autoenc': 7, 'dictlearn': 2, 'agg': 1}
Running C_dictlearn / kNN ...
  C_dictlearn / kNN                         Top-1 acc = 0.474  pred_dist={'kmeans': 15, 'gmm': 29, 'dbscan': 19, 'dictlearn': 7, 'autoenc': 7, 'agg': 1}
Running C_dictlearn / MLP ...
  C_dictlearn / MLP                         Top

In [7]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for key, res in clf_results.items():
    opt, model = key.split(' / ')
    rows.append({'Option': opt, 'Model': model, 'Top-1 Accuracy': round(res['accuracy'], 3)})

# Add baselines
rows.append({'Option': '—', 'Model': 'Always k-means',  'Top-1 Accuracy': round(baseline_kmeans, 3)})
rows.append({'Option': '—', 'Model': 'Always dbscan',   'Top-1 Accuracy': round(baseline_dbscan, 3)})
rows.append({'Option': '—', 'Model': 'Oracle (upper)',  'Top-1 Accuracy': 1.000})

summary_df = pd.DataFrame(rows)
print('=== Classification LOO-CV Results ===')
print(summary_df.sort_values('Top-1 Accuracy', ascending=False).to_string(index=False))

=== Classification LOO-CV Results ===
       Option          Model  Top-1 Accuracy
            — Oracle (upper)           1.000
A_handcrafted            kNN           0.590
A_handcrafted            MLP           0.564
B_autoencoder            kNN           0.538
B_autoencoder            MLP           0.513
  C_dictlearn            kNN           0.474
  C_dictlearn            MLP           0.449
            —  Always dbscan           0.269
            — Always k-means           0.179


## Detailed Analysis — Best Classifier (Option A)

In [8]:
# Pick the best model on Option A to inspect in detail
knn_a_res = clf_results['A_handcrafted / kNN']
mlp_a_res = clf_results['A_handcrafted / MLP']
best_a_res = knn_a_res if knn_a_res['accuracy'] >= mlp_a_res['accuracy'] else mlp_a_res
best_a_name = 'kNN' if knn_a_res['accuracy'] >= mlp_a_res['accuracy'] else 'MLP'

print(f'=== Best on Option A: {best_a_name} (acc={best_a_res["accuracy"]:.3f}) ===')
print('\nClassification report:')
print(classification_report(best_a_res['trues'], best_a_res['preds'], zero_division=0))

print('\nConfusion matrix (rows=true, cols=pred):')
labels = sorted(df_a['best_method'].unique())
cm = confusion_matrix(best_a_res['trues'], best_a_res['preds'], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df.to_string())

=== Best on Option A: kNN (acc=0.590) ===

Classification report:
              precision    recall  f1-score   support

         agg       0.00      0.00      0.00         5
     autoenc       0.67      0.67      0.67         9
      dbscan       0.75      0.71      0.73        21
   dictlearn       0.00      0.00      0.00         3
         gmm       0.72      0.81      0.76        26
      kmeans       0.33      0.29      0.31        14

    accuracy                           0.59        78
   macro avg       0.41      0.41      0.41        78
weighted avg       0.58      0.59      0.58        78


Confusion matrix (rows=true, cols=pred):
           agg  autoenc  dbscan  dictlearn  gmm  kmeans
agg          0        0       0          2    0       3
autoenc      0        6       0          0    2       1
dbscan       0        1      15          0    2       3
dictlearn    2        0       1          0    0       0
gmm          1        1       1          1   21       1
kmeans       

In [9]:
# ── Which datasets were mis-predicted? ────────────────────────────────────────
misses = [
    {
        'dataset_id': ids_a[i],
        'true':       best_a_res['trues'][i],
        'predicted':  best_a_res['preds'][i],
        'LSE_true_method':  df_a.loc[i, f'LSE_{best_a_res["trues"][i]}'],
        'LSE_pred_method':  df_a.loc[i, f'LSE_{best_a_res["preds"][i]}'],
    }
    for i in range(len(ids_a))
    if not best_a_res['correct_mask'][i]
]

miss_df = pd.DataFrame(misses)
if len(miss_df) > 0:
    miss_df['LSE_cost'] = (miss_df['LSE_true_method'] - miss_df['LSE_pred_method']).round(3)
    print(f'=== {len(miss_df)} mis-predictions ===')
    print(miss_df.to_string(index=False))
    print(f'\nMean LSE cost of errors: {miss_df["LSE_cost"].mean():.3f}')
    print(f'(= how much LSE is lost by picking the wrong method)')
else:
    print('Perfect prediction — no errors!')

=== 32 mis-predictions ===
 dataset_id      true predicted  LSE_true_method  LSE_pred_method  LSE_cost
      41004    kmeans    dbscan           0.6899           0.4163     0.274
       4153    kmeans       agg           0.7143           0.6000     0.114
       1465       agg dictlearn           0.5833           0.3333     0.250
      46879    kmeans       gmm           0.8200           0.8200     0.000
      47000       gmm   autoenc           0.7593           0.7130     0.046
      40997    dbscan    kmeans           0.7242           0.4464     0.278
         62    kmeans       gmm           1.0000           1.0000     0.000
         22       agg    kmeans           0.6400           0.4667     0.173
      45688    dbscan       gmm           1.0000           0.7500     0.250
         11       gmm       agg           0.7732           0.5979     0.175
       1551       agg dictlearn           0.6667           0.2917     0.375
        377       agg    kmeans           0.7000           0.

## Regression Variant — Predict All 6 LSE Values

In [10]:
knn_reg = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('reg',    KNeighborsRegressor(n_neighbors=best_k)),
])

mlp_reg = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('reg',    MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        max_iter=500,
        random_state=SEED,
    )),
])

print('Regression pipelines defined')

Regression pipelines defined


In [11]:
# ── Regression LOO-CV on Option A only (main approach) ────────────────────────
X_a_reg, Y_a_reg, _, _ = extract_Xy_reg(df_a)

reg_results = {}
for model_name, pipe in [('kNN', knn_reg), ('MLP', mlp_reg)]:
    print(f'Running regression {model_name} ...')
    res = loo_regress(pipe, X_a_reg, Y_a_reg, df_a)
    reg_results[model_name] = res
    print(f'  MAE mean: {res["mae_mean"]:.4f}')
    print(f'  MAE per method: ' + '  '.join(
        f'{m}={v:.3f}' for m, v in zip(METHOD_NAMES, res['mae_per_col'])
    ))
    print(f'  Argmax accuracy (predict best via argmax of predicted LSE): {res["argmax_accuracy"]:.3f}')
    print(f'  Mean expected LSE of predicted method: {res["expected_lse"]:.3f}')
    print()

print(f'Oracle expected LSE (upper bound): {oracle_lse:.3f}')
print(f'Always k-means expected LSE:       {always_kmeans_lse:.3f}')

Running regression kNN ...
  MAE mean: 0.1247
  MAE per method: kmeans=0.124  dbscan=0.125  agg=0.125  gmm=0.125  autoenc=0.120  dictlearn=0.129
  Argmax accuracy (predict best via argmax of predicted LSE): 0.590
  Mean expected LSE of predicted method: 0.706

Running regression MLP ...
  MAE mean: 0.1492
  MAE per method: kmeans=0.150  dbscan=0.166  agg=0.133  gmm=0.145  autoenc=0.157  dictlearn=0.143
  Argmax accuracy (predict best via argmax of predicted LSE): 0.333
  Mean expected LSE of predicted method: 0.677

Oracle expected LSE (upper bound): 0.767
Always k-means expected LSE:       0.618


## Ablation Summary — Option A vs B vs C

In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

abl_rows = []
for opt_name in ['A_handcrafted', 'B_autoencoder', 'C_dictlearn']:
    for model_name in ['kNN', 'MLP']:
        key  = f'{opt_name} / {model_name}'
        acc  = clf_results[key]['accuracy']
        abl_rows.append({'Option': opt_name, 'Model': model_name, 'Accuracy': acc})

abl_df = pd.DataFrame(abl_rows)
print('=== Ablation: Top-1 Classification Accuracy ===')
pivot = abl_df.pivot(index='Option', columns='Model', values='Accuracy').round(3)
print(pivot.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
w = 0.35
knn_accs = [abl_df.loc[(abl_df.Option == o) & (abl_df.Model == 'kNN'), 'Accuracy'].values[0]
            for o in ['A_handcrafted', 'B_autoencoder', 'C_dictlearn']]
mlp_accs = [abl_df.loc[(abl_df.Option == o) & (abl_df.Model == 'MLP'), 'Accuracy'].values[0]
            for o in ['A_handcrafted', 'B_autoencoder', 'C_dictlearn']]

ax.bar(x - w/2, knn_accs, w, label='kNN', color='steelblue')
ax.bar(x + w/2, mlp_accs, w, label='MLP', color='coral')
ax.axhline(baseline_kmeans, ls='--', color='gray', label=f'Always k-means ({baseline_kmeans:.2f})')
ax.axhline(oracle_acc,      ls=':',  color='black', label='Oracle (1.0)')
ax.set_xticks(x)
ax.set_xticklabels(['A: Hand-crafted', 'B: Autoencoder', 'C: Dict Learning'])
ax.set_ylabel('Top-1 Accuracy (LOO-CV)')
ax.set_title('Meta-learner ablation — feature representation comparison')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
plt.tight_layout()
fig_path = os.path.join(FIGS_DIR, 'ablation_accuracy.png')
fig.savefig(fig_path, dpi=120)
plt.close()
print(f'Figure saved → {fig_path}')

=== Ablation: Top-1 Classification Accuracy ===
Model            MLP    kNN
Option                     
A_handcrafted  0.564  0.590
B_autoencoder  0.513  0.538
C_dictlearn    0.449  0.474
Figure saved → c:\MLResearch\outputs\figures\ablation_accuracy.png


## Save Best Models

In [13]:
# ── Train final models on full dataset (no held-out fold) ─────────────────────

# ---- Classifier ----
best_clf_pipe = knn_clf if knn_a_res['accuracy'] >= mlp_a_res['accuracy'] else mlp_clf

final_clf = clone(best_clf_pipe)
final_clf.fit(X_a, y_a)

clf_path = os.path.join(MODELS_DIR, 'meta_clf_optA.pkl')
with open(clf_path, 'wb') as f:
    pickle.dump({'pipeline': final_clf, 'feature_cols': feat_cols_a}, f)
print(f'Classifier saved → {clf_path}  ({best_a_name})')

# ---- Regressor ----
best_reg_name = 'kNN' if reg_results['kNN']['mae_mean'] <= reg_results['MLP']['mae_mean'] else 'MLP'
best_reg_pipe = knn_reg if best_reg_name == 'kNN' else mlp_reg

final_reg = clone(best_reg_pipe)
valid_mask = ~np.isnan(Y_a_reg).any(axis=1)
final_reg.fit(X_a[valid_mask], Y_a_reg[valid_mask])

reg_path = os.path.join(MODELS_DIR, 'meta_reg_optA.pkl')
with open(reg_path, 'wb') as f:
    pickle.dump({
        'pipeline': final_reg,
        'feature_cols': feat_cols_a,
        'lse_cols': LSE_COLS,
        'method_names': METHOD_NAMES,
    }, f)
print(f'Regressor  saved → {reg_path}  ({best_reg_name})')

# ---- Save best B and C classifiers for ablation/app use ----
for letter, full_key_prefix, df_opt in [
    ('B', 'B_autoencoder', df_b),
    ('C', 'C_dictlearn',   df_c),
]:
    X_opt, y_opt, feat_opt, _ = extract_Xy_clf(df_opt)
    knn_acc_opt = clf_results[f'{full_key_prefix} / kNN']['accuracy']
    mlp_acc_opt = clf_results[f'{full_key_prefix} / MLP']['accuracy']
    best_pipe_opt = knn_clf if knn_acc_opt >= mlp_acc_opt else mlp_clf
    final_clf_opt = clone(best_pipe_opt)
    final_clf_opt.fit(X_opt, y_opt)
    path_opt = os.path.join(MODELS_DIR, f'meta_clf_opt{letter}.pkl')
    with open(path_opt, 'wb') as f:
        pickle.dump({'pipeline': final_clf_opt, 'feature_cols': feat_opt}, f)
    print(f'Saved opt{letter} clf → {path_opt}')

Classifier saved → c:\MLResearch\outputs\models\meta_clf_optA.pkl  (kNN)
Regressor  saved → c:\MLResearch\outputs\models\meta_reg_optA.pkl  (kNN)
Saved optB clf → c:\MLResearch\outputs\models\meta_clf_optB.pkl
Saved optC clf → c:\MLResearch\outputs\models\meta_clf_optC.pkl


## Final Summary

In [14]:
print('=' * 60)
print('PHASE 4 SUMMARY')
print('=' * 60)
print(f'  n_datasets        : {len(df_a)}')
print(f'  Best k (kNN)      : {best_k}')
print()
print('  CLASSIFICATION (LOO-CV Top-1 Accuracy)')
print(f'  Lower bound       : {baseline_kmeans:.3f}  (always k-means)')
print(f'  Upper bound       : {oracle_acc:.3f}  (oracle)')
for key in [
    'A_handcrafted / kNN', 'A_handcrafted / MLP',
    'B_autoencoder / kNN', 'B_autoencoder / MLP',
    'C_dictlearn / kNN',   'C_dictlearn / MLP',
]:
    acc = clf_results[key]['accuracy']
    print(f'  {key:35s}: {acc:.3f}')
print()
print('  REGRESSION (LOO-CV MAE + argmax accuracy, Option A)')
for model_name, res in reg_results.items():
    print(f'  {model_name:5s}  MAE={res["mae_mean"]:.4f}  '
          f'argmax_acc={res["argmax_accuracy"]:.3f}  '
          f'expected_LSE={res["expected_lse"]:.3f}')
print()
print(f'  Models saved to: {MODELS_DIR}')
print()
print('Phase 4 complete. Ready for Phase 5 (SHAP analysis).')

PHASE 4 SUMMARY
  n_datasets        : 78
  Best k (kNN)      : 1

  CLASSIFICATION (LOO-CV Top-1 Accuracy)
  Lower bound       : 0.179  (always k-means)
  Upper bound       : 1.000  (oracle)
  A_handcrafted / kNN                : 0.590
  A_handcrafted / MLP                : 0.564
  B_autoencoder / kNN                : 0.538
  B_autoencoder / MLP                : 0.513
  C_dictlearn / kNN                  : 0.474
  C_dictlearn / MLP                  : 0.449

  REGRESSION (LOO-CV MAE + argmax accuracy, Option A)
  kNN    MAE=0.1247  argmax_acc=0.590  expected_LSE=0.706
  MLP    MAE=0.1492  argmax_acc=0.333  expected_LSE=0.677

  Models saved to: c:\MLResearch\outputs\models

Phase 4 complete. Ready for Phase 5 (SHAP analysis).
